In [1]:
import os, pathlib

import keras
from keras import layers

<img src="images/banner.png" style="width: 100%;">

# NLP using RNNs Final Model Performances

## 1 Data Loading

In [3]:
from keras.utils import text_dataset_from_directory

data_dir = pathlib.Path('data/aclImdb')

train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

In [4]:
batch_size = 8

train_ds = text_dataset_from_directory(train_dir, batch_size=batch_size)
val_ds = text_dataset_from_directory(val_dir, batch_size=batch_size)
test_ds = text_dataset_from_directory(test_dir, batch_size=batch_size)

train_ds_no_labels = train_ds.map(lambda x, y: x)

Found 20000 files belonging to 2 classes.
Found 5000 files belonging to 2 classes.
Found 25000 files belonging to 2 classes.


## 2 Model Evaluation

### BoW Model

#### Vectorizer

In [5]:
max_tokens = 20_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
)
text_vectorization.adapt(train_ds_no_labels)

bag_of_words_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bag_of_words_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

#### Model Performance

In [6]:
bow_model = keras.models.load_model("models/bow_model.keras")
bow_acc = bow_model.evaluate(bag_of_words_test_ds)[1]
bow_acc

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 13s 4ms/step - accuracy: 0.8883 - loss: 0.2812


0.8883200287818909

### Bi-Gram Model

#### Vectorizer

In [9]:
max_tokens = 30_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="multi_hot",
    ngrams=2,
)
text_vectorization.adapt(train_ds_no_labels)

bigram_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
bigram_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

#### Model Performance

In [10]:
bigram_model = keras.models.load_model("models/bigram_model.keras")
bigram_acc = bigram_model.evaluate(bigram_test_ds)[1]
bigram_acc

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 12s 4ms/step - accuracy: 0.9020 - loss: 0.2504


0.9020400047302246

### LSTM

#### Vectorizer

In [11]:
max_length = 600
max_tokens = 30_000

text_vectorization = layers.TextVectorization(
    max_tokens=max_tokens,
    split="whitespace",
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(train_ds_no_labels)

sequence_train_ds = train_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_val_ds = val_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)
sequence_test_ds = test_ds.map(
    lambda x, y: (text_vectorization(x), y), num_parallel_calls=8
)

#### One-Hot Encoder

In [12]:
from keras import ops

class OneHotEncoding(keras.Layer):
    def __init__(self, depth, **kwargs):
        super().__init__(**kwargs)
        self.depth = depth

    def call(self, inputs):
        flat_inputs = ops.reshape(ops.cast(inputs, "int"), [-1])
        one_hot_vectors = ops.eye(self.depth)
        outputs = ops.take(one_hot_vectors, flat_inputs, axis=0)
        return ops.reshape(outputs, ops.shape(inputs) + (self.depth,))

one_hot_encoding = OneHotEncoding(max_tokens)

#### Model Performance

In [13]:
lstm_model = keras.models.load_model(
    "models/lstm_model.keras",
    custom_objects={'OneHotEncoding': OneHotEncoding}
)
lstm_acc = lstm_model.evaluate(sequence_test_ds)[1]
lstm_acc

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 22307s 7s/step - accuracy: 0.8322 - loss: 0.4006


0.8321999907493591

### LSTM with Embedding

In [14]:
lstm_with_embedding_model = keras.models.load_model("models/lstm_embedding.keras")
lstm_with_embedding_acc = lstm_with_embedding_model.evaluate(sequence_test_ds)[1]
lstm_with_embedding_acc

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 326s 104ms/step - accuracy: 0.8618 - loss: 0.3583


0.8617600202560425

### LSTM with Pre-trained Embedding

In [15]:
lstm_with_pretrained_embedding = keras.models.load_model(
    "models/lstm_pretrained_embedding.keras")
lstm_pretrained_embedding_acc = lstm_with_pretrained_embedding.evaluate(sequence_test_ds)[1]
lstm_pretrained_embedding_acc

3125/3125 ━━━━━━━━━━━━━━━━━━━━ 326s 104ms/step - accuracy: 0.8843 - loss: 0.3031


0.8843200206756592

## 3 Final Results

In [17]:
import pandas as pd

In [21]:
(pd.Series({'Bag-of-Words': bow_acc, 'Bi-gram': bigram_acc, 'LSTM': lstm_acc,
            'LSTM with Embedding': lstm_with_embedding_acc,
            'LSTM with Pretrained Embedding': lstm_pretrained_embedding_acc}
          ).to_frame('Accuracy').round(4)
           .sort_values('Accuracy', ascending=False)
)

,Accuracy
Bi-gram,0.9020
Bag-of-Words,0.8883
LSTM with Pretrained Embedding,0.8843
LSTM with Embedding,0.8618
LSTM,0.8322


<img src="images/banner-down.png" style="width: 100%;">